# Oracle DB 실시간 조회 대시보드 
- Gradio 화면에서 사용자가 카테고리를 선택하면, 그 즉시 **Oracle DB에 쿼리를 날려서** 결과
- Chapter02(Oracle 연동)와 Chapter03(Gradio)을 하나로 합치는 "미니 종합 실습"

### 💡 강의 포인트
- 지금까지는 "고정된 딕셔너리"나 "미리 읽어둔 CSV"로 실습했지만, 이번엔 **버튼을 누를 때마다 실제 DB 쿼리가 새로 실행**됩니다. Chapter04(종합 프로젝트)의 축소판이라고 보면 됩니다.
- 매번 함수 안에서 DB에 접속하고 `with` 구문으로 닫는 패턴은 Chapter02에서 이미 배운 것을 그대로 재사용합니다.
- `DELIVERY_ORDERS` 테이블과 `.env` 설정이 되어 있어야 정상 동작합니다 (Chapter02 준비물과 동일).

> ⚠️ Chapter03에서 가장 난이도가 높은 예제입니다. Chapter02의 ex01(DB 연결), ex09(피벗테이블)를 먼저 복습하고 진행하는 걸 권장합니다.

In [8]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='Malgun Gothic')
mpl.rc('axes', unicode_minus=False)

load_dotenv()
USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")

In [9]:
def query_category(category):
    """
    선택한 카테고리에 해당하는 주문 데이터를 Oracle에서 조회하고,
    요약 통계표 + 가격 히스토그램을 함께 반환하는 함수.
    """
    # 사용자가 입력값을 직접 SQL 문자열에 끼워넣지 않도록
    # 바인드 변수(:category)를 사용해 안전하게 조회합니다. (SQL Injection 방지)
    query = "SELECT * FROM DELIVERY_ORDERS WHERE CATEGORY = :category"

    with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
        df = pd.read_sql(query, conn, params={"category": category})
    # with 블록을 벗어나는 순간 자동으로 연결이 닫힘

    if len(df) == 0:
        # 데이터가 없는 경우를 대비한 방어 코드
        empty_df = pd.DataFrame({"안내": ["해당 카테고리의 데이터가 없습니다."]})
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "데이터 없음", ha="center")
        return empty_df, fig

    # 요약 통계
    summary = df[["PRICE", "RATING"]].describe().round(1)

    # 가격 히스토그램
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df["PRICE"], bins=8, color="salmon", edgecolor="black")
    ax.set_title(f"'{category}' 카테고리 가격 분포 (n={len(df)}건)")
    ax.set_xlabel("가격")
    ax.set_ylabel("건수")

    return summary, fig

In [ ]:
with gr.Blocks(title="배달 주문 실시간 대시보드") as demo:
    gr.Markdown("##  카테고리별 주문 데이터 실시간 조회")
    gr.Markdown("카테고리를 선택하면 Oracle DB에서 즉시 데이터를 조회합니다.")

    with gr.Row():
        category_dropdown = gr.Dropdown(
            ["분식", "치킨", "피자", "중식", "일식"],
            label="카테고리 선택",
        )
        search_btn = gr.Button("조회하기")

    with gr.Row():
        summary_output = gr.Dataframe(label="요약 통계")
        plot_output = gr.Plot(label="가격 분포")

    search_btn.click(
        fn=query_category,
        inputs=category_dropdown,
        outputs=[summary_output, plot_output],
    )

demo.launch()

In [11]:

# ⚠️ 실습 팁: 노트북에서 Gradio를 계속 켜두면 포트가 쌓여서 다음 셀 실행이 꼬일 수 있습니다.
# 확인이 끝나면 아래처럼 데모를 꺼주는 습관을 들이세요.
demo.close()


Closing server running on port: 7860


### ❓ 생각해볼 질문
1. SQL 쿼리에 `:category`처럼 바인드 변수를 쓰지 않고, `f"WHERE CATEGORY = '{category}'"`처럼 문자열을 직접 끼워 넣으면 어떤 보안 문제가 생길 수 있을까? (SQL Injection 힌트)
2. 카테고리를 선택할 때마다 매번 새로 `oracledb.connect()`를 하는 지금 방식은, 만약 사용자가 버튼을 100번 누른다면 어떤 부담이 생길까? 이걸 개선하려면 어떤 방법이 있을지 생각해보기. (Chapter04에서 다룰 개념 예고)